## Aurora_V4 - kWh

#### <span style="color:red; font-weight:bold"> Instructions for Using this Jupyter Notebook:</span>
This Notebook is using end point value minus start point value of kWh to calculate the kWh within one fiscal year.

<span style="color:royalblue">(0) Install Python packages (one-time setup):</span>
- In terminal, run: **pip3 install scikit-learn**

<span style="color:royalblue">(1) Place **data_clean.py** and this Notebook in the same folder (already done)</span>

<span style="color:royalblue">(2) Place **aurora_v4.meter_info.csv,** **building_complex_master_sheet_fyXX.xlsx**, **special_meters.xlsx**, and the **raw data files** in the same directory (already done)</span> 
- **aurora_v4.meter_info.csv** is exported from the database
- **NOTE:** Filename of the raw data files should include the variable name (i.e., kwh)

<span style="color:royalblue">(3) Modify the **Parameters in Section 1** as needed before running this Notebook:</span> 
- Time range, FY, etc.

<span style="color:royalblue">(4) Run the Notebook:</span> 
- Set **Checked = False** and run it once.
- Check **bad_meters_plots_fyXX.pdf** located in **output_dir**.
- If there are any unusual periods in the meter reading data, record them in **special_meters.xlsx** located in **input_dir**.
- If no problem, set **Checked = True** and run the notebook again.

<span style="color:royalblue">**Use R² to define Bad Meters (R² < 0.9):**</span> 

- **R² (Coefficient of Determination) Definition:** Measures how closely the meter’s kWh readings follow a perfect linear increase (straight) line over time using linear regression.

- **R² Values and Meaning:**
    - R² ≥ 0.9: Excellent — the meter closely follows the expected linear trend.

    - 0.5 ≤ R² < 0.9: Moderate — the meter shows noticeable deviations; may have some irregular readings.

    - 0 < R² < 0.5: Poor — the meter data is highly irregular.

    - R² = 0: All values missing — no valid data.

    - R² = -1: Missing points at start or end of the fiscal year; scaling applied if enough points exist (> 5 months).

    - R² = -2: Stuck points at start or end of the fiscal year; scaling applied.
 
    - R² = -3: Meter restarts at least 1 time.

### 1. Parameters

In [ ]:
############ CHANGE PARAMETERS AS NEEDED #############

Checked = False#True#   # Set to True after you checked the bad_meters plot.
Insert = False   # Set to False if there's no building_complex_master_sheet_fyXX.xlsx for that fiscal year.

# Time Range: Select One Fiscal Year
start_time = "2024-07-01 00:00:00"  #2025-07-23 09:40:50
end_time = "2025-07-01 00:00:00"
#FY = "_fy25"
FY = ""

######################################################

In [ ]:

# Data Directories
input_dir = "../data/extracts/"  # directory for raw data files & other input files

output_dir = "../data/outputs/"  # directory for data outputs (different from input_dir)

plot_dir = "../data/outputs/plots/"  # directory for plot outputs


# Variable
var = 'kwh'


# Input Files
#var_file = input_dir + "aurora_v4."+var+".fy22_fy25.csv"  # data file 0
var_file = input_dir + "processed_kwh.csv" # data file 0
#var_rejects_file = input_dir + "aurora_v4."+var+"_rejects.fy22_fy25.csv"  # data file 1
var_rejects_file = input_dir + "harvest."+var+"_rejects.csv"  # data file 1

meter_info_file = input_dir + "harvest.meter_info.csv"  # contains all meter information
meter_issues_file = input_dir + "meter_issues.xlsx"  # records special meters that need to be corrected

######################################################
if Insert:
    insert_sheet = input_dir + "building_complex_master_sheet" + FY + ".xlsx"  # insert output kWh usage into this sheet
    sheet_name = "complex"
    target_col_idx = 10  # insert into column K "Net kWh" (Note: Column A's index = 0)
######################################################


# Output Files
meter_annual_csv = output_dir + "meter_annual_" + var + FY + ".csv"  # annual kwh usage for each meter
building_annual_csv = output_dir + "building_annual_" + var + FY + ".csv"  # annual kwh usage for each building


# Output Figure
bad_meters_plot = plot_dir + "bad_meters_plots" + FY + ".pdf"

#do i ignore these?
# Exclude: Meters of buildings equipped with PV and Student Health
meters_with_pv = ['bachman_hall_main', 'campus_ctr_main', 'dance_bldg_main', 'gartley_hall_main', 'warrior_rec_ctr_main']
meters_excluded = meters_with_pv + ['student_health_main']  # student_health data is in vitality_v5


# Valid Data Min Length
valid_len = 5*30*96  # A meter should have at least 5-months valid data within 1 fiscal year

# Parameters for data cleaning - no need to change for now
r2_threshold = 0.9

# If a meter restarts more than 5 times in a fiscal year, treat it as a Bad Meter
restarts_thres = 5


# Data Frequency
freq = '15min'

# Schema
schema = 'harvest'


### 2. Imports

In [3]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Alignment
import math

import data_clean as dc  # import self-defined module


### 3. Load Data

In [5]:
# Read var and var_rejects tables
df0 = pd.read_csv(var_file)
df1 = pd.read_csv(var_rejects_file)

### 4. Data Processing

In [6]:
# The meter_reading values of these two meters in the kwh_rejects table are switched, so switch them back
swap_map = {
    "hale_aloha_ilima_tower_cafe": "hale_aloha_ilima_tower_main",
    "hale_aloha_ilima_tower_main": "hale_aloha_ilima_tower_cafe"
}

df1['meter_name'] = df1['meter_name'].replace(swap_map)


In [7]:
# Concatenate df0 and df1 vertically
combined_df = pd.concat([df0, df1], ignore_index=True)

# Convert 'datetime' to datetime type (if not already)
combined_df['datetime'] = pd.to_datetime(combined_df['datetime'])

# Sort by meter_name and datetime
combined_df = combined_df.sort_values(by=['meter_name', 'datetime']).reset_index(drop=True)


In [8]:
# Pivot table with every meter be one column
pivoted_df = combined_df.pivot(index='datetime', columns='meter_name', values='meter_reading').reset_index()

# Fill missing timestamps
full_df = dc.fill_missing_timestamps(pivoted_df, freq)


##### <span style="color:royalblue">Filter Meters and Time Range:</span>

In [9]:
### Filter One: Retain only main meters and filter out sub and PV meters ###

# Step 1: Read meter info
meter_info = pd.read_csv(meter_info_file)

# Step 2: Exclude certain meters first
all_meters = meter_info['meter_name'].unique()
non_exc_meters = [m for m in all_meters if m not in meters_excluded]

# Step 3: Get all 'main' meters from non-PV meters
main_meters_no_exc = meter_info[
    (meter_info['end_use'] == 'main') & 
    (meter_info['meter_name'].isin(non_exc_meters))]['meter_name'].unique()

# Step 4: Filter full_df to keep only non-PV 'main' meters (plus datetime)
columns_to_keep = ['datetime'] + list(full_df.columns.intersection(main_meters_no_exc))
filtered_df = full_df[columns_to_keep]

# Set index as datetime
filtered_df.set_index('datetime', inplace=True)


In [10]:
# filtered_df.loc["2024-06-10":"2024-06-17", "cmore_hale_main"].plot()

In [11]:
# filtered_df.loc["2024-06-10":"2024-06-16", "cmore_hale_main"].to_csv("TEST.csv", index=True)

In [12]:
### Filter Two: Retain only the selected Fiscal Year Data ###

data = filtered_df.loc[start_time:end_time, :].copy()
data.index = pd.to_datetime(data.index)

# Initial Data Cleaning: Replace all 0s with NaN in the entire DataFrame 
data = data.replace(0, np.nan)

data.head(2)


,admin_serv_1,admin_serv_2_main,ag_engineering_main,ag_science_main_1,ag_science_main_2,andrews_amp_main,archtecture_main,bachman_hall_annex,biomedical_science_main_a,biomedical_science_main_b,...,sinclair_lib_main,softball_tennis_main,spalding_hall_main,st_john_plant_science_main,stan_sheriff_ctr_main_1,stan_sheriff_ctr_main_2,transportation_srvc_main,univ_high_school_3_main,webster_hall_main,wist_annex_1_main
datetime,,,,,,,,,,,,,,,,,,,,,
2024-07-01 00:00:00,1174633.0,311723.0,1652785.0,207452.0,7212345.0,1993.0,2750440.0,NaN,69533817.0,5604841.0,...,NaN,510966.0,8908370.0,29139680.0,28339593.0,16299641.0,149607.0,554290.0,4148872.0,522239.0
2024-07-01 00:15:00,1174636.0,311723.0,1652787.0,207475.0,7212385.0,1993.0,2750450.0,NaN,69533978.0,5604851.0,...,NaN,510970.0,8908403.0,29139741.0,28339607.0,16299647.0,149607.0,554292.0,4148877.0,522240.0


##### <span style="color:royalblue">Correct Special Meters:</span>
- See **meter_issues_file.xlsx** in input_dir for details.

In [ ]:
data_corrected = dc.apply_special_meter_corrections(data, meter_issues_file)

##### <span style="color:royalblue">Find Bad Meters:</span>

In [14]:
# Find bad meters (R2 < 0.9)
df_bad_meters, df_restarts = dc.find_bad_meters(data, r2_threshold)
df_bad_meters


,meter_name,r2,info
0,archtecture_main,-3,restart 5 times
1,bus_ad_shidler_main,-3,restart 2 times
2,crawford_hall_main,-3,restart 2 times
3,dkac_pool_main,-3,restart 2 times
4,hig_substation_2_main,-3,restart 2 times
5,hper_klum_gym,-3,restart 9 times
6,les_murakami_stadium_main,-3,restart 2 times
7,malama_1_2_ehso_main,-3,restart 13 times
8,paradise_palms_main,-3,restart 1 times
9,pbrc_main_a,-3,restart 2 times


##### <span style="color:royalblue">Plot Bad Meters with Info:</span>

In [15]:
# Plot bad meters
dc.plot_bad_meters(data, df_bad_meters, bad_meters_plot)


Bad meter plots saved to ../data/outputs/plots/bad_meters_plots_fy25.pdf


In [16]:
if not Checked:
    raise SystemExit("Execution stopped because Checked = False")


#### <span style="color:red">!!! Note: Check bad_meters_plots_fyXX.pdf first before run the following cells !!!</span>

### 5. Calculate (End - Start) Difference for Annual kWh

In [17]:
### Compute kWh difference between End point and Start point; Scaling applied for some bad meters ###

# Step 1: Compute differences
result_df = dc.compute_meter_differences(
    data, start_time, end_time,
    df_bad_meters, df_restarts,
    valid_len=valid_len,
    r2_threshold=r2_threshold,
    restarts_thres=restarts_thres,
)

# Step 2: Export all meters' differences -> annual kWh usage (CSV, rounded 1 decimal)
df_all = dc.export_meter_differences(result_df, meter_info_file, meter_annual_csv, var=var)

# Step 3: Export building-level differences -> annual kWh per building (CSV)
df_building_sum = dc.export_building_differences(df_all, building_annual_csv, var=var)



##### <span style="color:royalblue">Insert data into master_sheet:</span>

In [18]:
### Insert kWh difference (annual usage) data into the Master Sheet ####

if Insert:

    # Load workbook and sheet
    wb = load_workbook(insert_sheet)
    ws = wb[sheet_name]

    df_diff = df_building_sum

    # Create mapping: building_complex_name -> annual_{var}
    diff_dict = dict(zip(df_diff['building_complex_name'], df_diff[f'annual_{var}']))

    # Loop rows starting from row 3
    for row in ws.iter_rows(min_row=3):
        building = row[2].value    # Column C (index 2)
        if building in diff_dict:
            val = diff_dict[building]

            # Column K (0-indexed = 10)
            cell = row[target_col_idx]

            # Handle NaN
            if val is None or (isinstance(val, float) and math.isnan(val)):
                cell.value = None
            else:
                cell.value = val

            # Right alignment
            cell.alignment = Alignment(horizontal="right")

    # Save workbook
    wb.save(insert_sheet)
